# 🏥 SmartLiva: YOLO26 Model Training on Google Colab (Free GPU)

สมุดโค้ดสำหรับฝึกสอนโมเดล **YOLO26** ทางการแพทย์ด้วย GPU ฟรีบน Google Colab (T4 / A100)

---

### ⚡ ขั้นตอนที่ 1: ตรวจสอบ GPU และติดตั้ง Ultralytics YOLO26

In [ ]:
# 1. ตรวจสอบการเชื่อมต่อ GPU
!nvidia-smi

# 2. ติดตั้ง Ultralytics เวอร์ชันล่าสุดที่รองรับ YOLO26
!pip install -q ultralytics

import ultralytics
print("✅ Ultralytics version:", ultralytics.__version__)

### 📦 ขั้นตอนที่ 2: อัปโหลดชุดข้อมูลและแตกไฟล์ Zip

In [ ]:
# 1. อัปโหลดไฟล์ lesion_yolo_train.zip และ/หรือ fibrosis_yolo_cls.zip ผ่านเมนูไฟล์ทางซ้ายของ Colab หรือรันคำสั่งนี้
from google.colab import files
import zipfile, os

# แตกไฟล์ lesion_yolo_train.zip (ถ้ามี)
if os.path.exists('lesion_yolo_train.zip'):
    print("Unzipping lesion dataset...")
    with zipfile.ZipFile('lesion_yolo_train.zip', 'r') as z:
        z.extractall('.')
    print("✅ Lesion dataset extracted!")

# แตกไฟล์ fibrosis_yolo_cls.zip (ถ้ามี)
if os.path.exists('fibrosis_yolo_cls.zip'):
    print("Unzipping fibrosis dataset...")
    with zipfile.ZipFile('fibrosis_yolo_cls.zip', 'r') as z:
        z.extractall('.')
    print("✅ Fibrosis dataset extracted!")

### 🎯 ขั้นตอนที่ 3: ฝึกสอนโมเดล `YOLO26s 7-Class Focal Lesion Detector`

In [ ]:
from ultralytics import YOLO
import yaml

# สร้างไฟล์ dataset.yaml ใน Colab
dataset_cfg = {
    "path": "/content/lesion_yolo_train",
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "Hemangioma",
        1: "Cyst",
        2: "Calcification",
        3: "Metastasis",
        4: "HCC",
        5: "CCA",
        6: "FFC"
    }
}

with open('/content/lesion_yolo_train/dataset.yaml', 'w') as f:
    yaml.dump(dataset_cfg, f)

# โหลด YOLO26s และเริ่มการเทรนบน GPU
model_lesion = YOLO('yolo26s.pt')

results_lesion = model_lesion.train(
    data='/content/lesion_yolo_train/dataset.yaml',
    epochs=40,
    imgsz=512,
    batch=32,
    device=0,  # GPU
    project='runs/lesion_train',
    name='yolo26s_lesion_colab',
    degrees=10.0,
    translate=0.10,
    scale=0.15,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    mixup=0.1,
    verbose=True
)

print("🎉 Lesion Training Completed!")

### 🧬 ขั้นตอนที่ 4: ฝึกสอนโมเดล `YOLO26s-cls Fibrosis (F0–F4)`

In [ ]:
from ultralytics import YOLO

# โหลด YOLO26s-cls และเริ่มเทรนพังผืดตับ 5 ระยะ
model_fibrosis = YOLO('yolo26s-cls.pt')

results_fibrosis = model_fibrosis.train(
    data='/content/yolo_cls_dataset',
    epochs=40,
    imgsz=448,
    batch=32,
    device=0,  # GPU
    project='runs/fibrosis_cls',
    name='yolo26s_fibrosis_colab',
    verbose=True
)

print("🎉 Fibrosis Training Completed!")

### 📥 ขั้นตอนที่ 5: ดาวน์โหลดไฟล์น้ำหนัก (best.pt) กลับมายังเครื่อง

In [ ]:
from google.colab import files

# 1. ดาวน์โหลดโมเดลรอยโรค (นำไปวางใน weights/lesion/yolo26s_lesion_best.pt)
if os.path.exists('runs/lesion_train/yolo26s_lesion_colab/weights/best.pt'):
    files.download('runs/lesion_train/yolo26s_lesion_colab/weights/best.pt')

# 2. ดาวน์โหลดโมเดลพังผืดตับ (นำไปวางใน weights/fibrosis/yolo26s_fibrosis_cls_best.pt)
if os.path.exists('runs/fibrosis_cls/yolo26s_fibrosis_colab/weights/best.pt'):
    files.download('runs/fibrosis_cls/yolo26s_fibrosis_colab/weights/best.pt')